# PyTorch进程之间的共享 Tensor（CUDA IPC）

利用CUDA IPC技术，实现进程之间共享Tensor，避免了数据拷贝，也能降低显存的消耗。常见的使用场景：KV cache在节点内的数据共享、RL训/推共卡权重数据的共享。

包含两个用例：
1. 单个脚本启动多进程共享数据，包括显式方式传递信息；
2. 两个脚本一收一发，完成Tensor数据共享。

Author: kaiyuan

Email: kaiyuanxie@yeah.net



# 1 单脚本多进程示例

PyTorch下通过CUDA IPC在进程间传递GPU Tensor的示例。

说明

1. 推荐方式：使用 ``torch.multiprocessing`` + ``spawn``，通过 ``Queue`` / ``Pipe``
   直接传递 ``cuda`` Tensor。序列化时会走与pickle相同的reduction，底层对
   ``UntypedStorage`` 调用 ``_share_cuda_()`` / ``_new_shared_cuda()``，即 CUDA IPC。

2. 显式方式：手动从 ``TypedStorage._share_cuda_()`` 取出元数据，再在子进程里调用
   内部的 ``rebuild_cuda_tensor`` 还原Tensor（与 PyTorch 多进程序列化逻辑一致），便于对照到底传了哪些IPC字段。依赖非公开API，仅作学习参考。

运行前请确认本机已安装带CUDA的PyTorch，且 ``torch.cuda.is_available()`` 为 True。

Windows/Linux均需在子进程中使用 ``spawn``；CUDA IPC一般用于同一台机器上的进程。

若在NFS等网络盘上运行脚本时出现 ``SemLock`` / ``FileNotFoundError``，除本文件在 Linux下默认
``TMPDIR=/tmp`` 外，仍可在shell中 ``export TMPDIR=/tmp``再启动。

Queue示例采用**三个独立Queue**：设备号、tensor、回执分开传递。若在同一Queue里先后``put``
``int``、tensor、再 ``get`` 回执，在 ``spawn`` + 内部feeder线程下可能出现**取到对象类型与预期不符**（例如第一次 ``get`` 到 tensor、回执读到 ``0``）。分队列可从根上避免混序。

示例运行环境：

docker pull nvcr.io/nvidia/pytorch:26.01-py3


In [1]:
from __future__ import annotations

# 必须在 import multiprocessing 之前设置：NFS 上常见 SemLock FileNotFoundError
import os
import sys

if sys.platform.startswith("linux"):
    os.environ.setdefault("TMPDIR", "/tmp")

from multiprocessing.connection import Connection

import torch
import torch.multiprocessing as mp


def _cuda_startup(device_index: int = 0) -> None:
    """
    在子进程里、对 CUDA tensor 做 unpickle / rebuild **之前**调用。
    顺序：先 set_device，再 init；部分驱动下顺序反了会报 invalid device context。
    再做一次极小 tensor 分配，确保当前 device 上 context 已完全就绪。
    """
    torch.cuda.set_device(device_index)
    if hasattr(torch.cuda, "init"):
        torch.cuda.init()
    else:
        torch.cuda._lazy_init()  # type: ignore[attr-defined]
    _ = torch.empty((1,), dtype=torch.uint8, device=f"cuda:{device_index}")
    torch.cuda.synchronize()


def _ensure_spawn() -> None:
    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass


# ---------------------------------------------------------------------------
# 方式一：Queue 传递 CUDA Tensor（推荐，与日常训练 DataLoader 等一致）
# ---------------------------------------------------------------------------


def _receiver_queue(q_dev: mp.Queue, q_tensor: mp.Queue, q_ack: mp.Queue) -> None:
    raw = q_dev.get()
    if not isinstance(raw, int):
        raise TypeError(
            f"expected int CUDA device index on q_dev, got {type(raw).__name__}"
        )
    device_idx = raw
    _cuda_startup(device_idx)
    t: torch.Tensor = q_tensor.get()
    print(
        f"[receiver] 收到 tensor: shape={tuple(t.shape)} "
        f"device={t.device} dtype={t.dtype}"
    )
    print(f"[receiver] mean={float(t.mean()):.6f}")
    # 与发送方共享同一块显存；发送方仍持有引用时，此处原地修改对发送方可见
    t.add_(1.0)
    q_ack.put("ok")


def run_queue_demo() -> None:
    """主进程创建 Tensor 放入 Queue，子进程取出并原地加 1。"""
    if not torch.cuda.is_available():
        print("CUDA 不可用，跳过 Queue 示例。", file=sys.stderr)
        return

    _ensure_spawn()
    ctx = mp.get_context("spawn")
    q_dev: mp.Queue = ctx.Queue()
    q_tensor: mp.Queue = ctx.Queue()
    q_ack: mp.Queue = ctx.Queue()
    p = ctx.Process(target=_receiver_queue, args=(q_dev, q_tensor, q_ack))
    p.start()

    _cuda_startup(0)
    x = torch.randn(4, 5, device="cuda", dtype=torch.float32).detach()
    dev = int(x.device.index if x.device.index is not None else 0)
    print(f"[sender] 发送前 mean={float(x.mean()):.6f} device=cuda:{dev}")
    q_dev.put(dev)
    q_tensor.put(x)
    ack = q_ack.get()
    print(f"[sender] 子进程回执: {ack}")
    print(f"[sender] 接收端 add_(1) 后的 mean={float(x.mean()):.6f}")
    p.join()


# ---------------------------------------------------------------------------
# 方式二：显式 _share_cuda_ 元数据 + rebuild_cuda_tensor（对照内部实现）
# ---------------------------------------------------------------------------


def _receiver_explicit(conn: Connection) -> None:
    from torch.multiprocessing.reductions import rebuild_cuda_tensor

    _cuda_startup(0)
    payload = conn.recv()
    t = rebuild_cuda_tensor(*payload)
    print(
        f"[receiver/explicit] 还原 tensor: shape={tuple(t.shape)} "
        f"sum={float(t.sum()):.4f}"
    )
    conn.send("done")


def run_explicit_ipc_demo() -> None:
    """
    主进程手动调用 ``storage._share_cuda_()``，把与 ``reduce_tensor`` 相同的参数包
    发给子进程，用 ``rebuild_cuda_tensor`` 还原。发送后须保持原 tensor 存活直至
    子进程完成打开 IPC（此处用 Pipe 同步）。
    """
    if not torch.cuda.is_available():
        print("CUDA 不可用，跳过显式 IPC 示例。", file=sys.stderr)
        return

    from torch.multiprocessing.reductions import (
        StorageWeakRef,
        rebuild_cuda_tensor,
        shared_cache,
    )

    _ensure_spawn()
    ctx = mp.get_context("spawn")
    parent_conn, child_conn = ctx.Pipe()

    p = ctx.Process(target=_receiver_explicit, args=(child_conn,))
    p.start()

    _cuda_startup(0)
    tensor = torch.arange(12, dtype=torch.float32, device="cuda").view(3, 4).detach()
    storage = tensor._typed_storage()
    (
        device,
        handle,
        storage_size_bytes,
        storage_offset_bytes,
        ref_counter_handle,
        ref_counter_offset,
        event_handle,
        event_sync_required,
    ) = storage._share_cuda_()
    # 与 torch.multiprocessing.reductions.reduce_tensor 中 CUDA 分支一致
    shared_cache[handle] = StorageWeakRef(storage)

    payload = (
        type(tensor),
        tensor.size(),
        tensor.stride(),
        tensor.storage_offset(),
        type(storage),
        tensor.dtype,
        device,
        handle,
        storage_size_bytes,
        storage_offset_bytes,
        tensor.requires_grad,
        ref_counter_handle,
        ref_counter_offset,
        event_handle,
        event_sync_required,
    )
    parent_conn.send(payload)
    parent_conn.recv()  # 等子进程 rebuild 完成后再退出作用域
    p.join()


def main() -> None:
    print("=== 1) torch.multiprocessing.Queue 传递 CUDA Tensor ===\n")
    run_queue_demo()
    print("\n=== 2) 显式 _share_cuda_ + rebuild_cuda_tensor ===\n")
    run_explicit_ipc_demo()


if __name__ == "__main__":
    main()

=== 1) torch.multiprocessing.Queue 传递 CUDA Tensor ===

[sender] 发送前 mean=0.160294 device=cuda:0
[receiver] 收到 tensor: shape=(4, 5) device=cuda:0 dtype=torch.float32
[receiver] mean=0.160294
[sender] 子进程回执: ok
[sender] 接收端 add_(1) 后的 mean=1.160294

=== 2) 显式 _share_cuda_ + rebuild_cuda_tensor ===

[receiver/explicit] 还原 tensor: shape=(3, 4) sum=66.0000



# 2 跨脚本运行示例

跨脚本（两个独立 Python 进程）共享 CUDA Tensor 的示例。


运行前设置环境变量：# export TMPDIR=/tmp

运行方式（开两个终端）：
1) 先启动接收端：
   python cuda_ipc_cross_script_demo.py --mode receiver --host 127.0.0.1 --port 29531

2) 再启动发送端：
   python cuda_ipc_cross_script_demo.py --mode sender --host 127.0.0.1 --port 29531 --device 0

预期现象：
- 接收端收到 CUDA tensor 后执行 t.add_(1.0)
- 发送端在收到 ack 后，能看到本地 x 的均值同步变化（说明两进程共享同一块显存）


In [ ]:
from __future__ import annotations

import argparse
import os
import sys
import time
from multiprocessing.connection import Client, Listener
from typing import Any

if sys.platform.startswith("linux"):
    os.environ.setdefault("TMPDIR", "/tmp")

import torch
import torch.multiprocessing.reductions as mp_reductions


def _cuda_startup(device_index: int) -> None:
    torch.cuda.set_device(device_index)
    if hasattr(torch.cuda, "init"):
        torch.cuda.init()
    else:
        torch.cuda._lazy_init()  # type: ignore[attr-defined]
    _ = torch.empty((1,), dtype=torch.uint8, device=f"cuda:{device_index}")
    torch.cuda.synchronize()


def run_receiver(host: str, port: int) -> None:
    listener = Listener((host, port), authkey=b"cuda-ipc-demo")
    print(f"[receiver] listening at {host}:{port}")
    conn = listener.accept()
    print("[receiver] sender connected")

    first_msg: Any = conn.recv()
    if not isinstance(first_msg, int):
        raise TypeError(f"expected device index int, got {type(first_msg).__name__}")
    device_idx = first_msg
    _cuda_startup(device_idx)

    t: torch.Tensor = conn.recv()
    print(
        f"[receiver] got tensor: shape={tuple(t.shape)} device={t.device} "
        f"mean={float(t.mean()):.6f}"
    )
    t.add_(1.0)
    torch.cuda.synchronize()
    conn.send({"status": "ok", "receiver_mean": float(t.mean())})
    conn.close()
    listener.close()
    print("[receiver] done")


def run_sender(host: str, port: int, device: int) -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")

    _cuda_startup(device)
    x = torch.randn(4, 5, device=f"cuda:{device}", dtype=torch.float32).detach()
    before = float(x.mean())
    print(f"[sender] before send mean={before:.6f}, device={x.device}")

    conn = Client((host, port), authkey=b"cuda-ipc-demo")
    conn.send(device)
    conn.send(x)
    ack = conn.recv()
    torch.cuda.synchronize()
    after = float(x.mean())
    print(f"[sender] ack={ack}")
    print(f"[sender] after receiver add_ mean={after:.6f}")
    conn.close()

    delta = after - before
    print(f"[sender] mean delta={delta:.6f}")
    if delta > 0.5:
        print("[sender] shared CUDA memory verified")
    else:
        print("[sender] WARNING: delta is small; verify synchronization/environment")
    time.sleep(0.2)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Cross-script CUDA IPC tensor sharing demo")
    parser.add_argument("--mode", choices=["sender", "receiver"], required=True)
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=29531)
    parser.add_argument("--device", type=int, default=0)
    return parser.parse_args()


def main() -> None:
    # 显式初始化 torch 的 reducer，确保 Connection.send/recv 处理 CUDA tensor 时走 IPC 分支
    mp_reductions.init_reductions()
    args = parse_args()
    if args.mode == "receiver":
        run_receiver(args.host, args.port)
    else:
        run_sender(args.host, args.port, args.device)


if __name__ == "__main__":
    main()


输出内容：

```
[sender] before send mean=0.173753, device=cuda:0
[sender] ack={'status': 'ok', 'receiver_mean': 1.1737531423568726}
[sender] after receiver add_ mean=1.173753
[sender] mean delta=1.000000
[sender] shared CUDA memory verified


[receiver] listening at 127.0.0.1:29531
[receiver] sender connected
[receiver] got tensor: shape=(4, 5) device=cuda:0 mean=0.173753
[receiver] done

```